# Intent Classification and Search Indexing
## Imports

In [16]:
import json, re
import pandas as pd
import numpy as np
from collections import defaultdict
from typing import Dict
from sklearn.svm import LinearSVC
from nltk.stem import PorterStemmer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, accuracy_score, confusion_matrix

In [17]:
# load the json
with open('SCCU-test.json', encoding='utf-8') as f:
    raw = json.load(f)

utts = raw['assets']['utterances']
df = pd.DataFrame([{'text': u['text'], 'intent': u['intent']} for u in utts])
print(df.shape)
df.head()

(451, 2)


,text,intent
0,Email,MyProfile
1,eStatement,NoticesandStatements
2,Statements,NoticesandStatements
3,eStatements,NoticesandStatements
4,Notices,NoticesandStatements


## Preprocessing

In [18]:
from nltk.stem import PorterStemmer

# Initialize the stemmer
stemmer = PorterStemmer()

def stem_text(text: str) -> str:
    # Split the sentence into words, stem each word, and join them back
    return ' '.join(stemmer.stem(word) for word in text.split())

stop_words = {
    'the', 'a', 'an', 'and', 'or', 'but', 'in', 'on', 'at', 'to', 'for',
    'of', 'with', 'is', 'was', 'are', 'been', 'be', 'have', 'has', 'had',
    'do', 'does', 'did', 'will', 'would', 'could', 'should', 'may', 'might',
    'must', 'can', 'this', 'that', 'these', 'those', 'i', 'you', 'he', 'she',
    'it', 'we', 'they', 'which', 'who', 'when', 'where', 'why', 'how'
}

def remove_stop_words(text: str) -> str:
    # Remove common stop words from the string
    return ' '.join(word for word in text.split() if word not in stop_words)


In [19]:
from collections import defaultdict

# 1. Build the word frequency index from the dataframe
word_index = defaultdict(int)
for text in df['text']:
    for word in text.split():
        word_index[word] += 1

# 2. Filter out the stop words inline
filtered_index = {word: count for word, count in word_index.items() if word not in stop_words}

# Preview the top non-stop words
sorted_words = sorted(filtered_index.items(), key=lambda x: x[1], reverse=True)
print("Top 10 words in index after removing stop words:")
print(sorted_words[:10])


Top 10 words in index after removing stop words:
[('loan', 46), ('pay', 22), ('account', 21), ('payment', 20), ('my', 19), ('what', 16), ('want', 15), ('transfer', 14), ('deposit', 11), ('bill', 11)]


### Inverted Index

In [20]:
# Initialize inverted index
inverted_index = defaultdict(list)

for idx, row in df.iterrows():
    # df['text'] already has stopword removal and stemming applied
    words = row['text'].split()
    for word in words:
        if idx not in inverted_index[word]:
            inverted_index[word].append(idx)

# Display stats and preview of the inverted index
print(f"Total unique terms in Inverted Index: {len(inverted_index)}")
print("\nInverted Index Preview (first 10 terms):")
for term in list(inverted_index.keys())[:10]:
    print(f"  '{term}': {inverted_index[term]}")

Total unique terms in Inverted Index: 364

Inverted Index Preview (first 10 terms):
  'Email': [0, 12]
  'eStatement': [1]
  'Statements': [2, 5]
  'eStatements': [3]
  'Notices': [4, 5, 6]
  '&': [5, 39, 81, 111, 153]
  'and': [6, 82, 83, 110, 234, 235, 346, 406]
  'Statments': [6]
  'Document': [7]
  'Documents': [8]


### Inverted Index Prediction

In [21]:
def predict_with_index(query):
    # preprocess query matching training preprocessing sequence
    clean_query = query.lower().strip()
    clean_query = re.sub(r'\s+', ' ', clean_query)
    clean_query = remove_stop_words(clean_query)
    query_stemmed = stem_text(clean_query)
    words = query_stemmed.split()
    
    if not words:
        print(f">> {query}\n  No search terms after stopword removal.")
        return
        
    intent_counts = defaultdict(int)
    
    # Retrieve document matches for each word
    for word in words:
        if word in inverted_index:
            for doc_id in inverted_index[word]:
                intent = df.iloc[doc_id]['intent']
                intent_counts[intent] += 1
                
    if not intent_counts:
        print(f">> {query}\n  No matches found in inverted index.")
        return
        
    # Rank intents
    sorted_intents = sorted(intent_counts.items(), key=lambda x: x[1], reverse=True)
    
    print(f">> {query} (using Inverted Index)\n")
    total_matches = sum(intent_counts.values())
    for rank, (intent, count) in enumerate(sorted_intents[:5], 1):
        print(f"  {rank}. {intent:<35} {count} matches ({count/total_matches*100:.1f}%)")

# Test prediction with inverted index
predict_with_index('transfer money to savings')
print()
predict_with_index('pay my bill')
print()
predict_with_index('nearest atm location')

>> transfer money to savings (using Inverted Index)

  1. TransferMoney                       7 matches (38.9%)
  2. CrossAccountTransfers               6 matches (33.3%)
  3. TalkativeHandOff                    2 matches (11.1%)
  4. PaymentCalender                     1 matches (5.6%)
  5. AccountInformation                  1 matches (5.6%)

>> pay my bill (using Inverted Index)

  1. BillPay                             20 matches (38.5%)
  2. LoanPayOff                          17 matches (32.7%)
  3. ChangeLoanDueDate                   3 matches (5.8%)
  4. MyProfile                           3 matches (5.8%)
  5. OnlineMobileBanking                 2 matches (3.8%)

>> nearest atm location (using Inverted Index)

  1. ATMLocator                          1 matches (100.0%)


## tfidf + logistic regression pipeline

In [22]:
# cleaning
df['text'] = df['text'].str.strip().str.lower()
df['text'] = df['text'].apply(lambda x: re.sub(r'\s+', ' ', x))

# drop rows with no alphanumeric content (e.g. "_", "-", ",")
mask = df['text'].str.contains(r'[a-z0-9]', regex=True)
print(f'dropping {(~mask).sum()} special-char rows')
df = df[mask].copy()

# drop dupes
df = df.drop_duplicates(subset=['text', 'intent']).reset_index(drop=True)

# remove stop words before stemming
df['text'] = df['text'].apply(remove_stop_words)

# apply stemming to each utterance
df['text'] = df['text'].apply(stem_text)

print(f'clean dataset: {len(df)} rows, {df.intent.nunique()} intents')
df.head(10)

dropping 37 special-char rows
clean dataset: 387 rows, 43 intents


,text,intent
0,email,MyProfile
1,estat,NoticesandStatements
2,statement,NoticesandStatements
3,estat,NoticesandStatements
4,notic,NoticesandStatements
5,notic & statement,NoticesandStatements
6,notic statment,NoticesandStatements
7,document,NoticesandStatements
8,document,NoticesandStatements
9,enrol set,MyProfile


In [23]:
# Filter out classes with only 1 sample
counts = df['intent'].value_counts()
df = df[df['intent'].isin(counts[counts >= 2].index)].reset_index(drop=True)

# 80/20 split
train, test = train_test_split(df, test_size=0.2, stratify=df['intent'], random_state=42)
print(f'train={len(train)}  test={len(test)}')

train=309  test=78


In [24]:
# tfidf + logistic regression pipeline
pipe = Pipeline([
    ('tfidf', TfidfVectorizer(ngram_range=(1, 2), sublinear_tf=True, norm=None)),
    ('clf',   LogisticRegression(C=5, max_iter=1000, random_state=42,))
])

pipe.fit(train['text'], train['intent'])
preds = pipe.predict(test['text'])

print('accuracy:', round(accuracy_score(test['intent'], preds), 4))

accuracy: 0.8718


In [25]:
print(classification_report(test['intent'], preds, zero_division=0))

                              precision    recall  f1-score   support

          AccountInformation       1.00      1.00      1.00         4
                    Accounts       1.00      1.00      1.00         1
                   ApplyLoan       0.83      1.00      0.91         5
             AuthorizedUsers       1.00      1.00      1.00         2
                     BillPay       1.00      0.67      0.80         3
                  BoatRVLoan       1.00      1.00      1.00         3
                CardControls       1.00      1.00      1.00         3
           ChangeLoanDueDate       1.00      1.00      1.00         2
                CheckDeposit       1.00      0.33      0.50         3
                   ContactUs       0.50      1.00      0.67         1
       CrossAccountTransfers       1.00      1.00      1.00         2
          ExternalACHAccount       1.00      1.00      1.00         2
                        Help       1.00      0.50      0.67         2
              HomeE

In [26]:
# quick predict function
def predict(query):
    # preprocess query matching training preprocessing sequence
    clean_query = query.lower().strip()
    clean_query = re.sub(r'\s+', ' ', clean_query)
    clean_query = remove_stop_words(clean_query)
    query_stemmed = stem_text(clean_query)
    probs = pipe.predict_proba([query_stemmed])[0]
    top   = np.argsort(probs)[::-1][:5]
    print(f'>> {query}\n')
    for r, i in enumerate(top, 1):
        print(f'  {r}. {pipe.named_steps["clf"].classes_[i]:<35} {probs[i]*100:.1f}%')

predict('transfer money to savings')
print()
predict('pay my bill')
print()
predict('nearest atm location')

>> transfer money to savings

  1. TransferMoney                       99.6%
  2. CrossAccountTransfers               0.0%
  3. ApplyLoan                           0.0%
  4. PaymentCalender                     0.0%
  5. TalkativeHandOff                    0.0%

>> pay my bill

  1. BillPay                             94.5%
  2. LoanPayOff                          4.6%
  3. MyProfile                           0.1%
  4. PayaPerson                          0.0%
  5. NoticesandStatements                0.0%

>> nearest atm location

  1. ATMLocator                          98.4%
  2. MyProfile                           0.1%
  3. ContactUs                           0.1%
  4. ApplyLoan                           0.1%
  5. NoticesandStatements                0.1%


In [27]:
# export to excel
train_out = train.assign(split='train')
test_out  = test.assign(split='test', predicted=preds,
                        correct=test['intent'].values == preds)

with pd.ExcelWriter('SCCU_tfidf_results.xlsx', engine='openpyxl') as w:
    df.to_excel(w, sheet_name='all', index=False)
    train_out.to_excel(w, sheet_name='train', index=False)
    test_out.to_excel(w, sheet_name='test_predictions', index=False)

print('saved SCCU_tfidf_results.xlsx')

PermissionError: [Errno 13] Permission denied: 'SCCU_tfidf_results.xlsx'

In [34]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GridSearchCV

# 1. Convert text to numbers (Vectorization)
vectorizer = TfidfVectorizer(ngram_range=(1, 2), sublinear_tf=True, norm=None)
X_train = vectorizer.fit_transform(train['text'])
X_test = vectorizer.transform(test['text'])

# 2. Define the model and the parameters to test
model = LogisticRegression(max_iter=1000, random_state=42)

param_grid = {
    'penalty': ['l2'],
    'C': np.logspace(-4, 4, 20),
    'solver': ['lbfgs', 'newton-cg', 'sag', 'saga'],
    'max_iter': [100, 1000, 2500]
}
# 3. Initialize Grid Search (uses 3-fold cross-validation for simplicity)
grid = GridSearchCV(estimator=model, param_grid=param_grid, cv=3)

# 4. Run the search on the training data
grid.fit(X_train, train['intent'])

# 5. Print the best settings found
print("Best Parameters:", grid.best_params_)
print(f"Best CV Accuracy: {grid.best_score_ * 100:.2f}%")

# 6. Test the best model on test data
best_model = grid.best_estimator_
test_accuracy = best_model.score(X_test, test['intent'])
print(f"Test Set Accuracy: {test_accuracy * 100:.2f}%")


c:\Users\anandha.kumar\OneDrive - ClaySys Technologies\Desktop\Code Base\Global search\.venv\Lib\site-packages\sklearn\model_selection\_split.py:812: UserWarning: The least populated class in y has only 2 members, which is less than n_splits=3.
  warnings.warn(
c:\Users\anandha.kumar\OneDrive - ClaySys Technologies\Desktop\Code Base\Global search\.venv\Lib\site-packages\sklearn\linear_model\_logistic.py:1403: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', l1_ratio set to a float between 0 and 1 instead of penalty='elasticnet', and C=np.inf instead of penalty=None.
  warnings.warn(
c:\Users\anandha.kumar\OneDrive - ClaySys Technologies\Desktop\Code Base\Global search\.venv\Lib\site-packages\sklearn\linear_model\_logistic.py:599: ConvergenceWarning: lbfgs failed to converge after

Best Parameters: {'C': np.float64(206.913808111479), 'max_iter': 100, 'penalty': 'l2', 'solver': 'lbfgs'}
Best CV Accuracy: 65.05%
Test Set Accuracy: 85.90%


c:\Users\anandha.kumar\OneDrive - ClaySys Technologies\Desktop\Code Base\Global search\.venv\Lib\site-packages\sklearn\linear_model\_logistic.py:1403: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', l1_ratio set to a float between 0 and 1 instead of penalty='elasticnet', and C=np.inf instead of penalty=None.
  warnings.warn(


In [35]:
import joblib

# 'pipe' is your trained Pipeline containing TfidfVectorizer and LogisticRegression
joblib.dump(pipe, 'intent_classifier.joblib')


['intent_classifier.joblib']